# Router Component - Hop Count Classification

**Model folder:** `components/router/models/qwen3_4b`

**Component:** Router (Stage A)

**Execution:** Kaggle or Colab

## 1. Environment + Versions

In [ ]:
import sys
import json
import random
import re
from pathlib import Path
from datetime import datetime

import torch
import transformers

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Config

In [ ]:
MODEL_NAME = "qwen3_4b_instruct_2507"
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

CONFIG = {
    "seed": 42,
    "model_id": MODEL_ID,
    "model_name": MODEL_NAME,
    "data_path": "/kaggle/input/metaqa-questions/refined_1hop.txt",
    "data_path_2hop": "/kaggle/input/metaqa-questions/refined_2hop.txt",
    "data_path_3hop": "/kaggle/input/metaqa-questions/refined_3hop.txt",
    "output_dir": Path("/kaggle/working/runs/router"),
    "run_id": datetime.now().strftime("%Y%m%d_%H%M%S"),
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # For classification, keep generation short + deterministic
    "max_new_tokens": 16,
    "temperature": 0.0,
    "top_p": 1.0,
    "do_sample": False,

    "approach": "few_shot_direct",
}

# Set seed for reproducibility
random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

print(f"Config: {json.dumps(CONFIG, indent=2, default=str)}")
print(f"Seed: {CONFIG['seed']}")
print(f"Approach: {CONFIG['approach']}")


## 3. Data Loading

In [ ]:
def load_questions(file_path: str):
    """Load questions from a text file (one per line)."""
    path = Path(file_path)
    if not path.exists():
        print(f"Warning: {file_path} not found. Using placeholder data.")
        return []

    with open(path, "r", encoding="utf-8") as f:
        questions = [line.strip() for line in f if line.strip()]
    return questions


# Load test questions (update paths when MetaQA dataset is uploaded)
questions_1hop = load_questions(CONFIG["data_path"])
questions_2hop = load_questions(CONFIG["data_path_2hop"])
questions_3hop = load_questions(CONFIG["data_path_3hop"])

print(f"Loaded {len(questions_1hop)} 1-hop questions")
print(f"Loaded {len(questions_2hop)} 2-hop questions")
print(f"Loaded {len(questions_3hop)} 3-hop questions")

# Example questions for testing if files not found
if not questions_1hop:
    questions_1hop = ["What genre is Inception?"]
if not questions_2hop:
    questions_2hop = ["Who directed movies starring Brad Pitt?"]
if not questions_3hop:
    questions_3hop = ["What languages are films that share directors with The Matrix?"]

## 4. Model Setup

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print(f"Loading model: {CONFIG['model_id']}")

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_id"])

# Safety: ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

if CONFIG["device"] == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_id"],
        torch_dtype="auto",
        device_map="auto",
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_id"],
        torch_dtype=torch.float32,
    ).to(CONFIG["device"])

model.eval()
print("Model loaded successfully")


## 5. Router Function

In [ ]:
PROMPT_TEMPLATE = """You are an assistant that determines how many hops a question requires.
- 1-hop: one direct fact answers the question.
- 2-hop: two linked facts are needed.
- 3-hop: three linked facts are needed.

Question: {question}
Reasoning:
Output:

Example 1:
Question: What genre is Inception?
Reasoning: Inception -> genre (one fact)
Output: 1

Example 2:
Question: Who directed movies starring Brad Pitt?
Reasoning: Brad Pitt -> movies -> directors (two facts)
Output: 2

Example 3:
Question: The director of Avatar directed which other movies?
Reasoning: Avatar -> director -> other movies (two facts)
Output: 2

Example 4:
Question: What release years are movies starring Emma Watson?
Reasoning: Emma Watson -> movies -> release years (two facts)
Output: 2

Example 5:
Question: What languages are films that share directors with The Matrix?
Reasoning: The Matrix -> director -> other films -> languages (three facts)
Output: 3

Example 6:
Question: Who acted in movies whose directors also directed Inception?
Reasoning: Inception -> director -> other movies -> actors (three facts)
Output: 3

Notes:
- Use step-by-step reasoning before the final answer.
- The final line must be only 1, 2, or 3.
"""


def build_prompt(question: str) -> str:
    return PROMPT_TEMPLATE.format(question=question)


def classify_hop_count(question: str, model, tokenizer, device: str) -> int:
    prompt = build_prompt(question)

    # Use Qwen3 chat formatting (recommended in the model card)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=CONFIG["temperature"],
            top_p=CONFIG["top_p"],
            do_sample=CONFIG["do_sample"],
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only newly generated tokens (more reliable than slicing by char)
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    response = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    try:
        # harmless even though this model won't emit <think> blocks
        clean_response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()

        # Often models put the answer on the first line; keep it tight
        first_line = clean_response.split("\n")[0].strip()

        match = re.search(r"[123]", first_line)
        if match:
            return int(match.group())

        lower_resp = first_line.lower()
        if "one" in lower_resp or "1-hop" in lower_resp:
            return 1
        if "two" in lower_resp or "2-hop" in lower_resp:
            return 2
        if "three" in lower_resp or "3-hop" in lower_resp:
            return 3

        numbers = re.findall(r"\d+", first_line)
        if numbers:
            return max(1, min(3, int(numbers[0])))

        print(f"Warning: No hop count in response: '{response[:80]}'. Defaulting to 2.")
        return 2

    except Exception as e:
        print(f"Error parsing response '{response[:50]}...': {e}. Defaulting to 2.")
        return 2


## 6. Run / Inference

In [ ]:
def classify_batch(questions, model, tokenizer, device: str, show_progress: bool = True):
    results = []
    for i, q in enumerate(questions):
        if show_progress and (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{len(questions)}...")
        hop = classify_hop_count(q, model, tokenizer, device)
        results.append(hop)
    return results


all_questions = []
all_expected = []

if questions_1hop:
    all_questions.extend(questions_1hop)
    all_expected.extend([1] * len(questions_1hop))
if questions_2hop:
    all_questions.extend(questions_2hop)
    all_expected.extend([2] * len(questions_2hop))
if questions_3hop:
    all_questions.extend(questions_3hop)
    all_expected.extend([3] * len(questions_3hop))

print(f"Running inference on {len(all_questions)} questions...")
print(f"  1-hop: {len(questions_1hop)}, 2-hop: {len(questions_2hop)}, 3-hop: {len(questions_3hop)}")
print()

predictions = classify_batch(all_questions, model, tokenizer, CONFIG["device"])

## 7. Evaluation + Metrics

In [ ]:
# Calculate metrics
correct = sum(1 for p, e in zip(predictions, all_expected) if p == e)
accuracy = correct / len(predictions) if predictions else 0.0

hop_1_correct = sum(1 for p, e in zip(predictions, all_expected) if e == 1 and p == e)
hop_1_total = sum(1 for e in all_expected if e == 1)
hop_1_acc = hop_1_correct / hop_1_total if hop_1_total > 0 else 0.0

hop_2_correct = sum(1 for p, e in zip(predictions, all_expected) if e == 2 and p == e)
hop_2_total = sum(1 for e in all_expected if e == 2)
hop_2_acc = hop_2_correct / hop_2_total if hop_2_total > 0 else 0.0

hop_3_correct = sum(1 for p, e in zip(predictions, all_expected) if e == 3 and p == e)
hop_3_total = sum(1 for e in all_expected if e == 3)
hop_3_acc = hop_3_correct / hop_3_total if hop_3_total > 0 else 0.0

from collections import Counter
confusion = Counter()
for p, e in zip(predictions, all_expected):
    confusion[(e, p)] += 1

metrics = {
    "overall_accuracy": accuracy,
    "total_questions": len(predictions),
    "correct_predictions": correct,
    "hop_1_accuracy": hop_1_acc,
    "hop_1_total": hop_1_total,
    "hop_2_accuracy": hop_2_acc,
    "hop_2_total": hop_2_total,
    "hop_3_accuracy": hop_3_acc,
    "hop_3_total": hop_3_total,
    "seed": CONFIG["seed"],
    "model": CONFIG["model_id"],
    "run_id": CONFIG["run_id"],
}

print("\n" + "=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"Overall Accuracy: {accuracy:.4f} ({correct}/{len(predictions)})")
print(f"1-hop Accuracy:   {hop_1_acc:.4f} ({hop_1_correct}/{hop_1_total})")
print(f"2-hop Accuracy:   {hop_2_acc:.4f} ({hop_2_correct}/{hop_2_total})")
print(f"3-hop Accuracy:   {hop_3_acc:.4f} ({hop_3_correct}/{hop_3_total})")

print("\n--- Confusion Matrix (Expected -> Predicted) ---")
print("     Pred->   1    2    3")
for expected in [1, 2, 3]:
    row = f"True {expected}:"
    for predicted in [1, 2, 3]:
        count = confusion.get((expected, predicted), 0)
        row += f" {count:4d}"
    print(row)

## 8. Save Artifacts

In [ ]:
output_dir = CONFIG["output_dir"] / CONFIG["run_id"]
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

with open(output_dir / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

notes = f"""Router Component Run - {CONFIG['run_id']}

Model: {CONFIG['model_id']}
Seed: {CONFIG['seed']}

Results:
- Overall Accuracy: {metrics['overall_accuracy']:.4f}
- 1-hop Accuracy: {metrics['hop_1_accuracy']:.4f} ({metrics['hop_1_total']} questions)
- 2-hop Accuracy: {metrics['hop_2_accuracy']:.4f} ({metrics['hop_2_total']} questions)
- 3-hop Accuracy: {metrics['hop_3_accuracy']:.4f} ({metrics['hop_3_total']} questions)

What to check:
- Review per-hop accuracy to identify which hop counts are harder to classify
- Check detailed_results.json for specific misclassification patterns
"""

with open(output_dir / "notes.md", "w") as f:
    f.write(notes)

detailed_results = [
    {
        "question": q,
        "expected_hop": e,
        "predicted_hop": p,
        "correct": p == e,
    }
    for q, e, p in zip(all_questions, all_expected, predictions)
]

with open(output_dir / "detailed_results.json", "w") as f:
    json.dump(detailed_results, f, indent=2, ensure_ascii=False)

output_timestamp = datetime.now().isoformat()
model_outputs = [
    {
        "question": q,
        "model_answer": p,
        "model_name": CONFIG["model_id"],
        "run_id": CONFIG["run_id"],
        "timestamp": output_timestamp,
    }
    for q, p in zip(all_questions, predictions)
]

with open(output_dir / "model_outputs.json", "w") as f:
    json.dump(model_outputs, f, indent=2, ensure_ascii=False)

confusion_data = {
    "matrix": {f"true_{e}_pred_{p}": confusion.get((e, p), 0) for e in [1, 2, 3] for p in [1, 2, 3]},
    "labels": [1, 2, 3],
}
with open(output_dir / "confusion.json", "w") as f:
    json.dump(confusion_data, f, indent=2)

print(f"\nArtifacts saved to: {output_dir}")
print("- metrics.json")
print("- config.json")
print("- notes.md")
print("- detailed_results.json")
print("- model_outputs.json")
print("- confusion.json")